# FASE INGENIERIA DE CARACTERISTICAS Y LIMPIEZA (TRANSFORM) FOOD DELIVERY

Durante la etapa de transformación se realizaron procesos de limpieza, estandarización e integración de datos con el objetivo de mejorar la calidad de la información y preparar un conjunto de datos consistente para las etapas de análisis y modelado.

In [34]:
## clonar o actualizar repositorio github en colab

from pathlib import Path
import os

ruta_proyecto = Path("/content/etl-python-analisis-food-delivery")

# Si el repositorio no existe, lo clona
if not ruta_proyecto.exists():
    %cd /content
    !git clone https://github.com/dannyturpo-data/etl-python-analisis-food-delivery.git

# Si ya existe, actualiza cambios
else:
    %cd /content/etl-python-analisis-food-delivery
    !git pull origin main

/content/etl-python-analisis-food-delivery
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (2/2), done.
remote: Total 19 (delta 2), reused 2 (delta 2), pack-reused 17 (from 1)
Unpacking objects: 100% (19/19), 19.14 MiB | 2.14 MiB/s, done.
From https://github.com/dannyturpo-data/etl-python-analisis-food-delivery
 * branch            main       -> FETCH_HEAD
   5c4264e..ddbdea5  main       -> origin/main
Updating 5c4264e..ddbdea5
Updating files: 100% (25/25), done.
Fast-forward
 data/procesada/df_master.csv                       |  68387 ++++++++++
 .../restaurant_menus_parte_1.csv                   |      0
 .../restaurant_menus_parte_10.csv                  |      0
 .../restaurant_menus_parte_2.csv                   |  12850 +-
 .../restaurant_menus_parte_3.csv                   |      0
 .../restaurant_menus_parte_4.csv                   |   9466 +-
 .../restaurant_menus_parte_5.csv                   |   6506 +-
 .../restaurant_menus_parte_6.csv                 

In [35]:
## configuracion

import sys

if str(ruta_proyecto) not in sys.path:
    sys.path.append(str(ruta_proyecto))

## importando librerias

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

## funciones propias

from src.exploracion_df import exploratorio_df
from src.limpieza import limpieza
from src.split_direcciones import separar_direccion

## LIMPIEZA Y TRANSFORMACION DE DF RESTAURANTS.CSV

Se aplicaron diversas transformaciones sobre el archivo restaurants.csv, entre las que destacan:

Mapeo de categorías de precio: Se normalizaron los valores de la variable de precios mediante un proceso de mapeo, convirtiendo diferentes representaciones en una escala uniforme. Esto permitió reducir inconsistencias y facilitar el análisis comparativo entre restaurantes.

Desanidado de la dirección: La columna que contenía la dirección completa fue dividida en tres nuevas variables:

Calle
Ciudad
Estado_Zip

Estandarización de datos: Se uniformizó el formato de diversas columnas, corrigiendo diferencias en mayúsculas, espacios en blanco y otros problemas de formato con el fin de garantizar consistencia en los registros.

In [37]:
## dataframe df_restaurantes

df_restaurantes = pd.read_csv('/content/etl-python-analisis-food-delivery/data/raw/restaurants.csv/restaurants.csv')

In [38]:
df_restaurantes_backup = df_restaurantes.copy()
df_restaurantes_backup.head(5)

,id,position,name,score,ratings,category,price_range,full_address,zip_code,lat,lng
0,1,19,PJ Fresh (224 Daniel Payne Drive),NaN,NaN,"Burgers, American, Sandwiches",$,"224 Daniel Payne Drive, Birmingham, AL, 35207",35207,33.562365,-86.830703
1,2,9,J' ti`'z Smoothie-N-Coffee Bar,NaN,NaN,"Coffee and Tea, Breakfast and Brunch, Bubble Tea",NaN,"1521 Pinson Valley Parkway, Birmingham, AL, 35217",35217,33.583640,-86.773330
2,3,6,Philly Fresh Cheesesteaks (541-B Graymont Ave),NaN,NaN,"American, Cheesesteak, Sandwiches, Alcohol",$,"541-B Graymont Ave, Birmingham, AL, 35204",35204,33.509800,-86.854640
3,4,17,Papa Murphy's (1580 Montgomery Highway),NaN,NaN,Pizza,$,"1580 Montgomery Highway, Hoover, AL, 35226",35226,33.404439,-86.806614
4,5,162,Nelson Brothers Cafe (17th St N),4.7,22.0,"Breakfast and Brunch, Burgers, Sandwiches",NaN,"314 17th St N, Birmingham, AL, 35203",35203,33.514730,-86.811700


### MAPEO DE PRECIOS:

La columna **`rango_de_precios`** contiene símbolos (`$`, `$$`, `$$$`, `$$$$`). Utilice un diccionario y la función `.map()` o `.replace()` para transformarlos a texto legible:

- `$` → **Económico**
- `$$` → **Moderadamente caro**
- `$$$` → **Caro**
- `$$$$` → **Muy caro**

In [39]:
## mapeo de precios
## diccionario de transformacion

mapeo_precios = {
    "$": "Económico",
    "$$": "Moderadamente caro",
    "$$$": "Caro",
    "$$$$": "Muy caro"
}

## aplicar el mapeo a la variable
## map() convierte los valores que no encuentra en el diccionario a NaN
## si existen valores nulos completamos.
df_restaurantes_backup["price_range"] = df_restaurantes_backup["price_range"].map(mapeo_precios).fillna("No Registra")


In [40]:
# valores unicos en variable price_range

df_restaurantes_backup["price_range"].unique()

array(['Económico', 'No Registra', 'Moderadamente caro', 'Caro',
       'Muy caro'], dtype=object)

In [41]:
df_restaurantes_backup.head(5)

,id,position,name,score,ratings,category,price_range,full_address,zip_code,lat,lng
0,1,19,PJ Fresh (224 Daniel Payne Drive),NaN,NaN,"Burgers, American, Sandwiches",Económico,"224 Daniel Payne Drive, Birmingham, AL, 35207",35207,33.562365,-86.830703
1,2,9,J' ti`'z Smoothie-N-Coffee Bar,NaN,NaN,"Coffee and Tea, Breakfast and Brunch, Bubble Tea",No Registra,"1521 Pinson Valley Parkway, Birmingham, AL, 35217",35217,33.583640,-86.773330
2,3,6,Philly Fresh Cheesesteaks (541-B Graymont Ave),NaN,NaN,"American, Cheesesteak, Sandwiches, Alcohol",Económico,"541-B Graymont Ave, Birmingham, AL, 35204",35204,33.509800,-86.854640
3,4,17,Papa Murphy's (1580 Montgomery Highway),NaN,NaN,Pizza,Económico,"1580 Montgomery Highway, Hoover, AL, 35226",35226,33.404439,-86.806614
4,5,162,Nelson Brothers Cafe (17th St N),4.7,22.0,"Breakfast and Brunch, Burgers, Sandwiches",No Registra,"314 17th St N, Birmingham, AL, 35203",35203,33.514730,-86.811700


### DESANIDADO DE UBICACIONES:

Desanidado de Ubicaciones (Split): La columna dirección_completa contiene
la calle, ciudad, estado y código postal separados por comas. Utilice el método
.str.split(',', expand=True) para crear tres nuevas columnas: calle, ciudad y
estado_zip. Asegúrese de aplicar .str.strip() a la nueva columna ciudad para
quitar espacios ocultos.

In [42]:
## conteo de "," por registros en la variable "full_address"
df_restaurantes_backup["full_address"].str.split(",").str.len().value_counts(dropna=False)

,count
full_address,
4.0,61155
5.0,1544
NaN,453
6.0,296
7.0,12
8.0,4
9.0,3
10.0,1
12.0,1


Se observa que la mayoria de registros contiene 4 "," separables en la direccion completa. Pero tambien existe un registro que contiene hasta 12 "," en la direccion. Entonces para crear las nuevas columnas se debe tener en cuenta esta caracteristica de la data. Se contara desde la derecha, ya que la estructura de la direccion muestra que tiene esta forma "....., estado, zip", tambien se debe considerar que hay algunas direcciones que no contienen el zip.

In [43]:
df_restaurantes_backup[["calle", "ciudad", "estado_zip"]] = (
    df_restaurantes_backup["full_address"]
    .apply(separar_direccion)
)

### FILTRO DE CALIDAD:

Elimine los restaurantes que no tengan puntaje (score nulo) o
cuyo puntaje sea 0. Estandarice la columna category a minúsculas.

In [44]:
## cuantos registros se eliminaran

# Cantidad de puntajes nulos
puntajes_nulos = df_restaurantes_backup["score"].isna().sum()
print(f"Cantidad de puntajes nulos: {puntajes_nulos}")

# Cantidad de puntajes iguales a 0
puntajes_cero = (df_restaurantes_backup["score"] == 0).sum()
print(f"Cantidad de puntajes iguales a 0: {puntajes_cero}")

Cantidad de puntajes nulos: 28167
Cantidad de puntajes iguales a 0: 0


In [45]:
## Elimando registros invalidos

# Eliminar registros con score nulo
df_restaurantes_backup = df_restaurantes_backup.dropna(subset=["score"])

# Eliminar registros con score igual a 0
df_restaurantes_backup = df_restaurantes_backup[
    df_restaurantes_backup["score"] != 0
]

In [46]:
## cantidad de valores nulos por columna
df_restaurantes_backup.isnull().sum().to_frame("valores nulos").T

,id,position,name,score,ratings,category,price_range,full_address,zip_code,lat,lng,calle,ciudad,estado_zip
valores nulos,0,0,0,0,0,9,0,190,191,0,0,190,190,190


Eliminando el resto de valores nulos.

In [47]:
## eliminar el resto de valores nulos

# Cantidad de direcciones nulas
print(df_restaurantes_backup["full_address"].isna().sum())
# Eliminar registros sin dirección
df_restaurantes_backup = df_restaurantes_backup.dropna(subset=["full_address"])

# Cantidad de zip_code nulas
print(df_restaurantes_backup["zip_code"].isna().sum())
# Eliminar registros sin zip
df_restaurantes_backup = df_restaurantes_backup.dropna(subset=["zip_code"])

# Cantidad de categorias nulas
print(df_restaurantes_backup["category"].isna().sum())
# Eliminar registros sin categoria
df_restaurantes_backup = df_restaurantes_backup.dropna(subset=["category"])

190
1
9


### ESTANDARIZAR LA CATEGORIA:

Convertir a minusculas

In [48]:
## estandarizar la categoria a minusculas
df_restaurantes_backup["category"] = (
    df_restaurantes_backup["category"]
    .str.lower()
)

In [49]:
df_restaurantes_backup.info()

<class 'pandas.core.frame.DataFrame'>
Index: 35102 entries, 4 to 63467
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            35102 non-null  int64  
 1   position      35102 non-null  int64  
 2   name          35102 non-null  object 
 3   score         35102 non-null  float64
 4   ratings       35102 non-null  float64
 5   category      35102 non-null  object 
 6   price_range   35102 non-null  object 
 7   full_address  35102 non-null  object 
 8   zip_code      35102 non-null  object 
 9   lat           35102 non-null  float64
 10  lng           35102 non-null  float64
 11  calle         35102 non-null  object 
 12  ciudad        35102 non-null  object 
 13  estado_zip    35102 non-null  object 
dtypes: float64(4), int64(2), object(8)
memory usage: 4.0+ MB


In [50]:
df_restaurantes_backup.head(5)

,id,position,name,score,ratings,category,price_range,full_address,zip_code,lat,lng,calle,ciudad,estado_zip
4,5,162,Nelson Brothers Cafe (17th St N),4.7,22.0,"breakfast and brunch, burgers, sandwiches",No Registra,"314 17th St N, Birmingham, AL, 35203",35203,33.514730,-86.811700,314 17th St N,Birmingham,"AL, 35203"
6,7,27,Jinsei Sushi,4.7,63.0,"sushi, asian, japanese",Económico,"1830 29th Ave S, Birmingham, AL, 35209",35209,33.480440,-86.790440,1830 29th Ave S,Birmingham,"AL, 35209"
13,14,51,Panera (521 Fieldstown Road),4.6,44.0,"breakfast and brunch, salad, sandwich, family ...",Económico,"521 Fieldstown Road, Gardendale, AL, 35071",35071,33.651407,-86.819247,521 Fieldstown Road,Gardendale,"AL, 35071"
15,16,88,Jeni's Splendid Ice Cream (Pepper Place),5.0,20.0,"ice cream &amp; frozen yogurt, comfort food, d...",Caro,"219 29th St S, Birmingham, AL, 35233",35233,33.516600,-86.789950,219 29th St S,Birmingham,"AL, 35233"
18,19,30,Falafel Cafe,4.9,48.0,"middle eastern, mediterranean, vegetarian, gre...",Económico,"401 19th St S, Birmingham, AL, 35233",35233,33.508353,-86.803170,401 19th St S,Birmingham,"AL, 35233"


## LIMPIEZA DF RESTAURANT_MENUS.CSV

En el archivo restaurant_menu.csv se realizaron procesos de depuración de datos para mejorar la calidad de la información disponible.

Las principales transformaciones fueron:

Eliminación de registros con valores nulos en las variables relevantes.
Filtrado de registros para conservar únicamente aquellos restaurantes con una calificación (rating) superior a 100, descartando registros con poca información o baja representatividad.

Estas operaciones permitieron trabajar únicamente con datos completos y de mayor calidad.

In [51]:
# creando diccionarios de DataFrames, se guardan dentro de dfs

ruta_csv = Path("/content/etl-python-analisis-food-delivery/data/raw/restaurant-menus.csv")

dfs = {}

# Buscar los archivos y ordenarlos por el número de la parte
for archivo in sorted(
    ruta_csv.glob("*.csv"),
    key=lambda archivo: int(archivo.stem.split("_")[-1])
):
    # Guardar cada DataFrame en el diccionario
    # archivo.stem archivo son extension
    dfs[archivo.stem] = pd.read_csv(archivo)

In [52]:
# que DataFrame se crearon
dfs.keys()

dict_keys(['restaurant_menus_parte_1', 'restaurant_menus_parte_2', 'restaurant_menus_parte_3', 'restaurant_menus_parte_4', 'restaurant_menus_parte_5', 'restaurant_menus_parte_6', 'restaurant_menus_parte_7', 'restaurant_menus_parte_8', 'restaurant_menus_parte_9', 'restaurant_menus_parte_10'])

In [53]:
# tamaño de cada DataFrame

for nombre, df in dfs.items():
    print(nombre, df.shape)

restaurant_menus_parte_1 (83635, 5)
restaurant_menus_parte_2 (83635, 5)
restaurant_menus_parte_3 (83635, 5)
restaurant_menus_parte_4 (83635, 5)
restaurant_menus_parte_5 (83635, 5)
restaurant_menus_parte_6 (83635, 5)
restaurant_menus_parte_7 (83635, 5)
restaurant_menus_parte_8 (83635, 5)
restaurant_menus_parte_9 (83635, 5)
restaurant_menus_parte_10 (83635, 5)


In [ ]:
# se crea un nuevo diccionario con copias de cada DataFrame
"""
dfs_backup = {
    nombre: df.copy()
    for nombre, df in dfs.items()
}

for nombre, df in dfs_backup.items():
    print(nombre, df.shape)
"""

### LIMPIEZA
1. La columna price viene como texto (ej. "15,99 USD"). Cree una función lambda o
use .str.replace() para eliminar el texto " USD", reemplazar la coma por un punto,
y castear el resultado a formato decimal (float).
2. Elimine los ítems del menú con precio igual a 0.0 o nulos.
3. Reemplazar las descripciones nulas por "sin descripcion".
4. Estandarizar a minusculas las columnas category, name y description.

In [54]:
dfs["restaurant_menus_parte_1"].isnull().sum().to_frame("valores nulos").T

,restaurant_id,category,name,description,price
valores nulos,0,0,0,20737,0


In [55]:
dfs_menus_limpios = {}

for nombre, df in dfs.items():
    dfs_menus_limpios[nombre] = limpieza(df)

In [56]:
dfs_menus_limpios["restaurant_menus_parte_1"].isnull().sum().to_frame("valores nulos").T

,restaurant_id,category,name,description,price
valores nulos,0,0,0,0,0


In [57]:
for nombre, df in dfs_menus_limpios.items():
    print(nombre, df.shape)

restaurant_menus_parte_1 (80544, 5)
restaurant_menus_parte_2 (78887, 5)
restaurant_menus_parte_3 (78041, 5)
restaurant_menus_parte_4 (76671, 5)
restaurant_menus_parte_5 (79800, 5)
restaurant_menus_parte_6 (77406, 5)
restaurant_menus_parte_7 (81256, 5)
restaurant_menus_parte_8 (81595, 5)
restaurant_menus_parte_9 (82491, 5)
restaurant_menus_parte_10 (82108, 5)


## OPTIMIZACION Y CRUCE (CONTROL DE MEMORIA RAM):

1. Para no colapsar la memoria del servidor, el CEO solo quiere analizar los restaurantes
consolidados. Filtre df_restaurantes para quedarse únicamente con aquellos que
tengan más de 100 calificaciones (ratings).
2. Realice un INNER JOIN (pd.merge) entre su df_restaurantes filtrado y df_menus
utilizando el ID del restaurante. Guarde este cruce en un DataFrame llamado df_master.

In [58]:
df_restaurantes_backup["ratings"].dtypes

dtype('float64')

In [59]:
## Filtramos registros con ratings mayor a 100
df_restaurantes_filtrado = df_restaurantes_backup[
    df_restaurantes_backup["ratings"] > 100
]

In [60]:
df_restaurantes_filtrado.shape

(8670, 14)

### UNIR TODOS LOS ARCHIVOS DE MENUS

Consolidar las partes del csv en un solo DataFrame.
El conjunto de datos de menús se encontraba dividido en diez archivos independientes.

Para consolidar la información se realizó un proceso de unión (concatenación) de todos los archivos, generando un único conjunto de datos que reúne la totalidad de los registros de menús.

Posteriormente, este conjunto consolidado fue integrado con el archivo restaurants.csv previamente limpiado mediante una operación Merge, utilizando el identificador común de los restaurantes como clave de unión.

Como resultado, se obtuvo un único dataset que contiene tanto la información general de cada restaurante como los datos asociados a sus respectivos menús, constituyendo la base final para las fases de análisis y explotación de datos.

In [61]:
## consolidando en un solo DF
df_restaurantes_menus = pd.concat(
    dfs_menus_limpios.values(),
    ignore_index=True
)

In [62]:
df_restaurantes_menus.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 798799 entries, 0 to 798798
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   restaurant_id  798799 non-null  int64  
 1   category       798799 non-null  object 
 2   name           798799 non-null  object 
 3   description    798799 non-null  object 
 4   price          798799 non-null  float64
dtypes: float64(1), int64(1), object(3)
memory usage: 30.5+ MB


### INNER JOIN

In [63]:
## merge entre los DF y consolidar un solo master
df_master = pd.merge(
    df_restaurantes_filtrado,
    df_restaurantes_menus,
    left_on="id",
    right_on="restaurant_id",
    how="inner"
)

In [64]:
## eliminar columna id para evitar redundancia
df_master = df_master.drop(columns=["restaurant_id"])
df_master = df_master.drop(columns=["zip_code"])

## cambiar nombre a algunas variables
df_master = df_master.rename(columns={"id": "restaurant_id"})
df_master = df_master.rename(columns={"name_x": "name_restaurant"})
df_master = df_master.rename(columns={"category_x": "category_restaurant"})
df_master = df_master.rename(columns={"name_y": "name_menu"})
df_master = df_master.rename(columns={"category_y": "category_menu"})

In [65]:
df_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63021 entries, 0 to 63020
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   restaurant_id        63021 non-null  int64  
 1   position             63021 non-null  int64  
 2   name_restaurant      63021 non-null  object 
 3   score                63021 non-null  float64
 4   ratings              63021 non-null  float64
 5   category_restaurant  63021 non-null  object 
 6   price_range          63021 non-null  object 
 7   full_address         63021 non-null  object 
 8   lat                  63021 non-null  float64
 9   lng                  63021 non-null  float64
 10  calle                63021 non-null  object 
 11  ciudad               63021 non-null  object 
 12  estado_zip           63021 non-null  object 
 13  category_menu        63021 non-null  object 
 14  name_menu            63021 non-null  object 
 15  description          63021 non-null 

In [66]:
## exportar a CSV df_master

print("Exportando DataFrame maestro...")
print(f"Registros: {len(df_master):,}")
print(f"Columnas: {df_master.shape[1]}")

df_master.to_csv("df_master.csv", index=False)

print("Archivo exportado correctamente.")

Exportando DataFrame maestro...
Registros: 63,021
Columnas: 17
Archivo exportado correctamente.


Al finalizar la fase de transformación se obtuvo un conjunto de datos caracterizado por:

* Eliminación de registros inconsistentes o incompletos.
* Variables estandarizadas y con formatos homogéneos.
* Información geográfica estructurada.
* Consolidación de los archivos de menús en un único dataset.
* Integración de la información de restaurantes y menús mediante una unión relacional.
* Datos preparados visualización y construcción de modelos analíticos.